In [27]:
import ROOT
import numpy as np
import pandas as pd
 
ROOT.gStyle.SetOptFit(1111)    # aggiunge il label con chi2/ndf e probabilita e parametri se metti 0000 toglie i dati

#funzione modello
def model(x, par):
    p = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    return B + (C*(p-A))/((p-A)-C)

def model_derivative(p, par):
    return -par[2]**2 / ((p - par[0]) - par[2])**2

#leggere file excel 
xlsx ="esperienza2.xlsx"
df = pd.read_excel("esperienza2.xlsx", sheet_name= "Dati")
df = df.dropna(subset=["p", "q", "sigma_p", "sigma_q"])
df = df.to_numpy(dtype = 'float64')

#definizione variabili
p = df[:, 5]
print(p)
q = df[:, 6]

err_p = df[:, 8]

misurato = ~np.isnan(df[:, 2])     
err_p = df[:, 8]
err_q = np.where(misurato, df[:, 7], np.sqrt(df[:, 7]**2 ))



#minimizzazzione con ROOT
model_fit = ROOT.TF1('model', model, min(p), max(p), 3)
model_fit.SetParameters(0.0, 0.0, 0.0)
model_fit.SetParNames("A", "B", "C")

gr1 = ROOT.TGraphErrors(len(p), p, q, err_p, err_q)
gr1.SetTitle("Dati sperimentali e fit; p; q")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)                    
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)        
gr1.SetLineWidth(1)


fit_res = gr1.Fit(model_fit, "SR")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = model_fit.GetParameters()

errors = model_fit.GetParErrors()

chi2 = fit_res.Chi2()

err_qeff = np.sqrt(err_q**2 + ((model_derivative(p, par)) * err_p)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"B = {par[1]} ± {errors[1]}")
print(f"C = {par[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("AP")


leg1 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(model_fit, "Fit", "l")
leg1.Draw()

c1.Draw()
c1.SaveAs("fit_root.png")

[ 232.5  233.5  234.5  235.5  236.5  237.5  238.5  239.5  240.5  241.5
  242.5  243.5  244.5  245.5  246.5  247.5  248.5  249.5  250.5  251.5
  252.5  253.5  254.5  255.5  256.5  257.5  258.5  259.5  260.5  261.5
  262.5  263.5  264.5  265.5  267.5  269.5  271.5  273.5  275.5  277.5
  280.5  281.5  282.5  283.5  284.5  285.5  286.5  287.5  288.5  289.5
  290.5  291.5  292.5  293.5  294.5  295.5  297.5  299.5  301.5  306.5
  311.5  316.5  321.5  326.5  331.5  336.5  341.5  346.5  351.5  356.5
  361.5  366.5  371.5  376.5  381.5  386.5  391.5  396.5  401.5  406.5
  411.5  416.5  421.5  426.5  431.5  441.5  451.5  461.5  471.5  481.5
  491.5  501.5  511.5  521.5  531.5  541.5  551.5  561.5  571.5  581.5
  601.5  621.5  641.5  661.5  681.5  731.5  781.5  831.5  881.5  981.5
 1081.5 1181.5 1281.5 1381.5 1481.5 1631.5]
Fit status: 0
Parametri stimati:
A = 0.5122143492642242 ± 0.20100100267746607
B = 2.9805378068525363 ± 0.1474853040041194
C = 200.04176956915725 ± 0.08916606535161006
Chi2 = 1

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file fit_root.png has been created


In [28]:
err_R = 1
err_L = 1
err_x0 = 0.115470053837925

if np.abs(par[0]) < 3*(errors[0] + err_x0 + err_R + err_L):
        print("A è compatibile con 0")
else :
        print("A NON è compatibile con 0")

print("---")

if np.abs(par[1]) < 3*(errors[1] + err_x0  + err_L):
    print("B è compatibile con 0")
else :
    print("B NON è compatibile con 0")
 
 


A è compatibile con 0
---
B è compatibile con 0


In [29]:
q_attesi = np.array([model_fit.Eval(x_val) for x_val in p])
residuals_norm = (q - q_attesi) / err_qeff


#residuals_norm = np.delete(residuals_norm, [0,1, 10, 12, 14, 43, 44, 52])

#print(residuals_norm)

gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(q), max(q), 3)
gaussian_fit.SetParameters(1, 0, 1)
gaussian_fit.SetParNames("const", "mu", "sigma")
gaussian_fit.SetLineColor(ROOT.kRed + 1)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 20, min(residuals_norm), max(residuals_norm))
hist.Fill(residuals_norm)
hist.SetLineColor(ROOT.kAzure + 3)
hist.SetLineWidth(2)
hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
hist.GetXaxis().SetTitle("residuo normalizzato")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati")

r = hist.Fit(gaussian_fit, "S")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors = gaussian_fit.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"#mu = {parameters[1]} ± {errors[1]}")
print(f"#sigma = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")


c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

leg2 = ROOT.TLegend(0.13, 0.80, 0.35, 0.89)
leg2.AddEntry(hist, "Residui", "f")
leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
leg2.Draw()


c2.Draw()
c2.SaveAs("hist.png")


print()
print()

for i in np.arange(0,len(residuals_norm)):
    if residuals_norm[i] < -0.8:
        print (residuals_norm[i])
        print(i)

Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).


Fit status: 0
Parametri stimati:
const = 61.761899136594735 ± 8.027770831169923
#mu = 3.096020616127188 ± 0.2502443165391207
#sigma = 2.0927586603622523 ± 0.15243106041125765
Chi2 = 35.88267142788309


-14.529092457952826
62
-7.324279966755955
69
-3.0902944278737143
70
-5.679664600402049
72
-13.034244551042509
73
-22.896247155084648
74
-29.603112753461183
75
-50.12065240347059
76
-61.429399300641364
77
-13.310752190975276
78
-58.663302788339884
88
-7.633506222958914
94
-9.174543696131932
101
-3.35259051491552
105
-7.776849573212222
106
-3.8401475146168544
107
-10.919906740086935
109
-12.514835870502985
110
-7.063304661193168
111
-7.856873373175774
113
-2.2706760577329073
114
-8.550541220789638
115
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      35.8827
NDf                       =           10
Edm                       =  3.02132e-06
NCalls                    =          191
const                     =      61.7619   +/-   8.02777  

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2
Info in <TCanvas::Print>: png file hist.png has been created


In [30]:
if np.abs(parameters[0] - 1) < 3*errors[0]:
    print("const è compatibile con 1")
else :
    print("const NON è compatibile con 1")

print("---")

if np.abs(parameters[1]) < 3*errors[1]:
    print("mu è compatibile con 0")
else :
    print("mu NON è compatibile con 0")

print("---")

if np.abs(parameters[2] - 1) < 3*errors[2]:
    print("sigma è compatibile con 1")
else :
    print("sigma NON è compatibile con 1")

const NON è compatibile con 1
---
mu NON è compatibile con 0
---
sigma NON è compatibile con 1


In [31]:
def line(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

residuals = q - q_attesi
print(residuals)

line_fit = ROOT.TF1('line', line, min(p), max(p), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")
line_fit.SetLineColor(ROOT.kRed + 1)
line_fit.SetLineWidth(2)

gr2 = ROOT.TGraphErrors(len(p), p, residuals, np.zeros_like(residuals), err_qeff)
gr2.SetTitle("Residui in funzione di p; p; q - q_{att}")
gr2.SetMarkerStyle(20)
gr2.SetMarkerSize(0.7)
gr2.SetMarkerColor(ROOT.kAzure + 2)
gr2.SetLineColor(ROOT.kAzure + 3)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors = line_fit.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("AP")

leg3 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg3.AddEntry(gr2, "Residui", "pe")
leg3.AddEntry(line_fit, "Fit lineare", "l")
leg3.Draw()

c3.Draw()
c3.SaveAs("fit_line.png")

[ 1.12737945e+02  1.04900252e+02  1.32768192e+02  1.00183243e+02
  1.00269731e+02  9.64560696e+01  9.45364118e+01  8.56284020e+01
  8.74352523e+01  7.62663579e+01  7.24096140e+01  8.16731387e+01
  6.42931368e+01  6.67076077e+01  5.90771070e+01  7.06015183e+01
  6.90305014e+01  6.54957055e+01  5.98829540e+01  5.92488136e+01
  5.34042856e+01  5.59331446e+01  5.22960027e+01  5.49081687e+01
  4.89821744e+01  4.60304577e+01  4.22430874e+01  3.94148447e+01
  4.27886514e+01  4.27075786e+01  3.88460699e+01  4.26302755e+01
  3.85706345e+01  3.71403464e+01  3.51445567e+01  3.39890012e+01
  3.23231405e+01  3.23069981e+01  3.04242838e+01  3.13527269e+01
  2.63068958e+01  2.79822737e+01  2.65221768e+01  2.09902196e+01
  1.87160404e+01  2.33277898e+01  2.03246459e+01  2.12015897e+01
  2.29223164e+01  2.05421196e+01  2.05438458e+01  2.24264425e+01
  1.82476356e+01  2.04075197e+01  1.90047003e+01  2.04909402e+01
  1.91709934e+01  1.90256534e+01  1.55266126e+01  1.52426601e+01
  1.47802585e+01  8.45441

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c3
Info in <TCanvas::Print>: png file fit_line.png has been created


In [32]:
# residui normalizzati divisi perzona di p

limiti = [232, 260, 316, 340, 460, 700, 2000]

print("zona di p      N   media   errore sulla media   std")
for i in range(len(limiti) - 1):
    sel = (p >= limiti[i]) & (p < limiti[i+1]) 
    r = residuals_norm[sel]
    print(limiti[i], "-", limiti[i+1], "  ", len(r), "  ", round(np.mean(r), 2), "  ",
          round(np.std(r) / np.sqrt(len(r)), 2), "  ", round(np.std(r), 2))

r = residuals_norm[:8]
print("primi 8 punti:", len(r), "  ", round(np.mean(r), 2), "  ",
      round(np.std(r) / np.sqrt(len(r)), 2), "  ", round(np.std(r), 2))

#  punti con residuo oltre 2 sigma
print("punti con |residuo| > 2:")
for i in range(len(p)):
    if abs(residuals_norm[i]) > 2:
        print("  indice", i, " p =", p[i], " residuo normalizzato =", round(residuals_norm[i], 1),
              " residuo [mm] =", round(q[i] - q_attesi[i], 1))



zona di p      N   media   errore sulla media   std
232 - 260    28    3.37    0.23    1.24
260 - 316    33    4.1    0.16    0.9
316 - 340    5    2.48    4.02    9.0
340 - 460    21    -3.6    4.72    21.63
460 - 700    18    5.34    4.24    18.0
700 - 2000    11    -5.7    1.28    4.25
primi 8 punti: 8    3.46    0.78    2.22
punti con |residuo| > 2:
  indice 0  p = 232.5  residuo normalizzato = 9.3  residuo [mm] = 112.7
  indice 1  p = 233.5  residuo normalizzato = 3.1  residuo [mm] = 104.9
  indice 2  p = 234.5  residuo normalizzato = 2.8  residuo [mm] = 132.8
  indice 3  p = 235.5  residuo normalizzato = 2.3  residuo [mm] = 100.2
  indice 4  p = 236.5  residuo normalizzato = 2.5  residuo [mm] = 100.3
  indice 5  p = 237.5  residuo normalizzato = 2.4  residuo [mm] = 96.5
  indice 6  p = 238.5  residuo normalizzato = 2.5  residuo [mm] = 94.5
  indice 7  p = 239.5  residuo normalizzato = 2.8  residuo [mm] = 85.6
  indice 8  p = 240.5  residuo normalizzato = 3.1  residuo [mm] = 87.4
